In [32]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [33]:
df = pd.read_csv("online_retail.csv")
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [34]:
print(df.info()) 
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB
None
           Quantity         Price    Customer ID
count  1.067371e+06  1.067371e+06  824364.000000
mean   9.938898e+00  4.649388e+00   15324.638504
std    1.727058e+02  1.235531e+02    1697.464450
min   -8.099500e+04 -5.359436e+04   12346.000000
25%    1.000000e+00  1.250000e+00   13975.000000
50%    3.000000e+00  2.100000e+00   15255.000000
75%    1.000000e+01  4.150000e+00   1

In [35]:
# null analysis 

# calculating null count 
null_count = df.isnull().sum()
print(null_count)

# calculating null percentage
null_pct = (df.isnull().sum()/len(df)) * 100
print(null_pct)


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float64


In [36]:
# cleaning the data 

# dropping null values in customer ID
df = df[df['Customer ID'].notna()]
print(df.shape)

# dropping null values in Description

df = df[df['Description'].notna()]
print(df.shape)

df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(df.shape)

df = df[df['Quantity']>0]
print(df.shape)

df = df[df['Price'] > 0]
print(df.shape)

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)


(824364, 8)
(824364, 8)
(805620, 8)
(805620, 8)
(805549, 8)
(805549, 8)


In [37]:
# date range check

print(df['InvoiceDate'].min())
print(df['InvoiceDate'].max())

2009-12-01 07:45:00
2011-12-09 12:50:00


In [38]:
# Now let's define churn window

snapshot_date = df['InvoiceDate'].max()
churn_window_start = snapshot_date - pd.Timedelta(days=90)

print("Snapshot date:", snapshot_date)
print("Churn window start: ", churn_window_start)

Snapshot date: 2011-12-09 12:50:00
Churn window start:  2011-09-10 12:50:00


In [39]:
features_df = df[df['InvoiceDate'] < churn_window_start]
label_df = df[df['InvoiceDate'] >= churn_window_start]

print(features_df)
print(label_df)

       Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
859319  566220     23307  SET OF 60 PANTRY DESIGN CAKE CASES        120   
859320  566220     22965   3 TRADITIONAl BISCUIT CUTTERS  SET        24   
859321  566220     22989       SET 2 PANTRY DESIGN TEA TOWELS        24   
859322  566220     22980               PANTRY SCRUBBING BRUSH        48   
859323  566220     22622       BOX OF VINTAGE ALPHABET BLOCKS         6   

               InvoiceDate  Price  Customer ID         Country  
0      2009-12-01 07:45:00   6.95 